# Reconhecimento de Faces com a InsightFace

No notebook anterior, exploramos a detecção de rostos com a `insightface`: localizar onde há uma face na imagem, seus pontos de referência, e alinhar o recorte para um padrão fixo. Neste notebook, damos o próximo passo: o **reconhecimento facial**, que responde a uma pergunta diferente, não *onde* está um rosto, mas *quem* é essa pessoa.

O reconhecimento facial moderno não compara pixels diretamente. Em vez disso, cada rosto alinhado passa por uma rede neural (o `ArcFace`, dentro do mesmo pacote `buffalo_l` que já usamos) que o reduz a um vetor de **512 números**, chamado de *embedding*. Esse vetor funciona como uma impressão digital numérica do rosto: fotos da mesma pessoa, mesmo em ângulos e iluminações diferentes, geram vetores próximos entre si no espaço vetorial; fotos de pessoas diferentes geram vetores distantes. Antes de aplicá-lo, vamos ver esse vetor na prática, comparando embeddings de rostos parecidos e diferentes.

Depois, vamos simular um cenário comum em produtos de RH e segurança, o **onboarding de colaboradores** por biometria facial, cobrindo cinco etapas:
1. **Construindo o banco de faces**, extraindo o embedding de cada foto de um banco de imagens de colaboradores.
2. **Busca 1:N**, comparando uma selfie de consulta com todo o banco para descobrir quem é a pessoa.
3. **Espaço vetorial com t-SNE**, visualizando como os embeddings se agrupam por identidade.
4. **Verificação 1:1**, o fluxo real de onboarding: comparar a selfie contra a foto de referência correta.
5. **Exemplo negativo**, repetindo a verificação 1:1 com uma identidade errada, para ver o sistema recusar.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93
    !pip install opencv-contrib-python==5.0.0.93
    !pip install insightface
    !pip install onnxruntime
    !pip install scikit-learn
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Reaproveitamos o `FaceAnalysis` do notebook anterior, mas agora carregando também o módulo `recognition`, responsável por gerar os embeddings. Para reduzir os embeddings de 512 dimensões a um plano 2D navegável, usamos o `TSNE` do `scikit-learn`.

In [ ]:
from pathlib import Path

from insightface.app import FaceAnalysis
from sklearn.manifold import TSNE

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline

## 1. Embeddings e Reconhecimento Facial

Assim como no notebook anterior, o `FaceAnalysis` carrega o pacote `buffalo_l`. Desta vez, também precisamos do módulo `recognition`, que adiciona o `w600k_r50` (ArcFace) à detecção: para cada rosto encontrado, ele alinha o recorte internamente (o mesmo `norm_crop` que vimos na Seção 4 do notebook anterior) e devolve um vetor de 512 dimensões, disponível em `face.normed_embedding`.

Como o próprio nome indica, esse vetor já vem normalizado (norma euclidiana igual a 1), o que simplifica a comparação entre dois rostos: basta calcular o produto escalar entre os dois vetores, a **similaridade de cosseno**, um número entre -1 e 1 que mede o quão alinhados eles estão. Quanto mais próximo de 1, mais parecidos os rostos. Na prática, para pessoas diferentes esse valor costuma ficar perto de 0; para a mesma pessoa, acima de 0.4.

### 1.1. Carregando o modelo

Diferente do notebook anterior, agora incluímos `'recognition'` em `allowed_modules`, para carregar também o modelo de embeddings.

In [ ]:
app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection', 'recognition'])
app.prepare(ctx_id=0, det_size=(640, 640))

### 1.2. Um Embedding, na Prática

Na teoria, um embedding é só um vetor de 512 números. Para tornar isso concreto, vamos extrair esse vetor de duas fotos da mesma pessoa e de uma foto de uma pessoa diferente, e comparar os resultados.

Como pode haver mais de um rosto numa foto, criamos aqui a função `area_rosto()`, usada ao longo de todo o notebook para escolher sempre o rosto de maior caixa delimitadora.

In [ ]:
def area_rosto(face) -> float:
    """ Calcula a área da caixa delimitadora de um rosto """
    x1, y1, x2, y2 = face.bbox
    return (x2 - x1) * (y2 - y1)

def extrair_embedding(image_path: str) -> np.ndarray:
    """ Detecta o rosto principal de uma imagem e devolve seu embedding normalizado """
    img = cv2.imread(image_path)
    faces = app.get(img)
    rosto = max(faces, key=area_rosto)
    return rosto.normed_embedding

embedding_a = extrair_embedding('imagens/05/banco/Tony_Blair/Tony_Blair_0001.jpg')
embedding_b = extrair_embedding('imagens/05/banco/Tony_Blair/Tony_Blair_0002.jpg')
embedding_c = extrair_embedding('imagens/05/banco/Colin_Powell/Colin_Powell_0001.jpg')

print(f"Formato do vetor: {embedding_a.shape}")
print(f"Primeiros 8 valores: {embedding_a[:8].round(3)}")

Um vetor isolado não diz muita coisa: os números não têm um significado individual óbvio, não existe uma posição "tamanho do nariz" ou "cor dos olhos". O que importa é a posição relativa entre vetores. Rostos parecidos ficam próximos nesse espaço de 512 dimensões, rostos diferentes ficam distantes. Vamos medir essa proximidade com a similaridade de cosseno, comparando as duas fotos de Tony Blair entre si, e cada uma delas com a foto de Colin Powell.

In [ ]:
imagens_exemplo = [
    ('imagens/05/banco/Tony_Blair/Tony_Blair_0001.jpg', 'Tony Blair (A)'),
    ('imagens/05/banco/Tony_Blair/Tony_Blair_0002.jpg', 'Tony Blair (B)'),
    ('imagens/05/banco/Colin_Powell/Colin_Powell_0001.jpg', 'Colin Powell (C)'),
]

fig, eixos = plt.subplots(1, 3, figsize=(9, 3.5))

for eixo, (path, titulo) in zip(eixos, imagens_exemplo):
    eixo.imshow(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB))
    eixo.set_title(titulo)
    eixo.axis('off')

plt.tight_layout()
plt.show()

sim_mesma_pessoa = float(np.dot(embedding_a, embedding_b))
sim_pessoa_diferente = float(np.dot(embedding_a, embedding_c))

print(f"A x B (mesma pessoa):        similaridade={sim_mesma_pessoa:.3f}")
print(f"A x C (pessoas diferentes):  similaridade={sim_pessoa_diferente:.3f}")

plt.figure(figsize=(4, 3.5))
plt.bar(
    ['A x B\n(mesma pessoa)', 'A x C\n(pessoas diferentes)'],
    [sim_mesma_pessoa, sim_pessoa_diferente],
    color=['#2ca02c', '#d62728'],
)
plt.ylabel("Similaridade de cosseno")
plt.ylim(-0.1, 1)
plt.title("Distância no espaço de embeddings")
plt.show()

A similaridade entre as duas fotos de Tony Blair fica bem acima da similaridade entre Tony Blair e Colin Powell, mesmo sem nenhuma das duas fotos de Blair serem idênticas entre si (poses, expressões e iluminação diferentes). É exatamente essa separação, embeddings da mesma pessoa próximos, de pessoas diferentes distantes, que sustenta tanto a busca 1:N quanto a verificação 1:1 que vamos construir a seguir.

## 2. Construindo o Banco de Faces

Toda solução de reconhecimento facial depende de um banco de rostos conhecidos para comparar. No nosso cenário de onboarding, esse banco representa o cadastro de colaboradores já verificados: fotos de cada pessoa, cada uma reduzida ao seu embedding.

A pasta `imagens/05/banco` reúne fotos de quatro pessoas, dez fotos cada, organizadas em uma subpasta por identidade, um recorte do [LFW (Labeled Faces in the Wild)](https://vis-www.cs.umass.edu/lfw/), o dataset acadêmico clássico de reconhecimento facial.

### 2.1. Listando as imagens

In [ ]:
banco_dir = Path('imagens/05/banco')
image_paths = sorted(banco_dir.glob('*/*.jpg'))

pessoas = sorted({p.parent.name for p in image_paths})

print(f"{len(image_paths)} imagens no banco, de {len(pessoas)} pessoas: {', '.join(pessoas)}")

### 2.2. Extraindo os embeddings

Para cada imagem, detectamos os rostos e guardamos o embedding do rosto principal, reaproveitando a função `area_rosto()` definida na Seção 1.2, útil caso alguma foto tenha mais de um rosto ao fundo. O nome da pessoa vem do nome da subpasta.

In [ ]:
embeddings_banco = []
nomes_banco = []
paths_banco = []

for image_path in image_paths:
    img = cv2.imread(str(image_path))
    faces = app.get(img)

    if not faces:
        print(f"Nenhum rosto encontrado em {image_path}")
        continue

    rosto_principal = max(faces, key=area_rosto)

    embeddings_banco.append(rosto_principal.normed_embedding)
    nomes_banco.append(image_path.parent.name)
    paths_banco.append(image_path)

embeddings_banco = np.array(embeddings_banco)

print(f"{len(embeddings_banco)} embeddings extraídos, dimensão {embeddings_banco.shape[1]}")

### 2.3. Amostra do banco

Antes de seguir, uma amostra de duas fotos por pessoa, para conferir visualmente o banco que acabamos de processar.

In [ ]:
amostras_por_pessoa = 2

fig, eixos = plt.subplots(len(pessoas), amostras_por_pessoa, figsize=(2.5 * amostras_por_pessoa, 2.8 * len(pessoas)))

for linha, pessoa in enumerate(pessoas):
    paths_pessoa = [p for p in paths_banco if p.parent.name == pessoa][:amostras_por_pessoa]

    for coluna, path in enumerate(paths_pessoa):
        img = cv2.imread(str(path))
        eixos[linha, coluna].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        eixos[linha, coluna].set_xticks([])
        eixos[linha, coluna].set_yticks([])

    eixos[linha, 0].set_ylabel(pessoa.replace('_', ' '))

plt.tight_layout()
plt.show()

## 3. Busca no Banco (1:N)

Com o banco pronto, vamos simular uma **busca de identificação (1:N)**: dada uma foto de consulta, comparar seu embedding contra *todos* os embeddings do banco e descobrir qual pessoa é a mais parecida. É o tipo de busca usada, por exemplo, para checar se um rosto capturado por uma câmera já está cadastrado em algum sistema, sem saber de antemão de quem se trata.

### 3.1. Carregando a imagem de consulta

In [ ]:
consulta_path = 'imagens/05/banco/consulta.jpg'

img_consulta = cv2.imread(consulta_path)

plt.figure(figsize=(4, 5))
plt.imshow(cv2.cvtColor(img_consulta, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

### 3.2. Extraindo o embedding da consulta

In [ ]:
faces_consulta = app.get(img_consulta)
print(f"{len(faces_consulta)} rosto(s) detectado(s) na consulta")

rosto_consulta = max(faces_consulta, key=area_rosto)
embedding_consulta = rosto_consulta.normed_embedding

### 3.3. Comparando com o banco inteiro

Como os embeddings já vêm normalizados, a similaridade de cosseno entre a consulta e cada rosto do banco se resume a um produto escalar. Fazemos isso de uma vez só, como uma multiplicação de matriz por vetor, e ordenamos os resultados da maior para a menor similaridade.

In [ ]:
similaridades = embeddings_banco @ embedding_consulta

top_k = 5
indices_top = np.argsort(-similaridades)[:top_k]

for indice in indices_top:
    print(f"{nomes_banco[indice]:<20} similaridade={similaridades[indice]:.3f}  ({paths_banco[indice].name})")

### 3.4. Apresentando os resultados

A imagem de consulta, à esquerda, e as cinco fotos do banco mais parecidas com ela, em ordem de similaridade.

In [ ]:
fig, eixos = plt.subplots(1, top_k + 1, figsize=(2.6 * (top_k + 1), 3.2))

eixos[0].imshow(cv2.cvtColor(img_consulta, cv2.COLOR_BGR2RGB))
eixos[0].set_title("Consulta")
eixos[0].axis('off')

for posicao, indice in enumerate(indices_top, start=1):
    img_banco = cv2.imread(str(paths_banco[indice]))
    eixos[posicao].imshow(cv2.cvtColor(img_banco, cv2.COLOR_BGR2RGB))
    eixos[posicao].set_title(f"{nomes_banco[indice].replace('_', ' ')}\nsim={similaridades[indice]:.2f}", fontsize=9)
    eixos[posicao].axis('off')

plt.tight_layout()
plt.show()

As cinco fotos mais similares pertencem todas à mesma pessoa. É essa concentração de similaridade em torno de uma única identidade, e não a pontuação isolada de uma única foto, que dá confiança a uma identificação numa busca 1:N.

## 4. Espaço Vetorial dos Embeddings com t-SNE

Os embeddings vivem em um espaço de 512 dimensões, impossível de visualizar diretamente. O [t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) (*t-distributed Stochastic Neighbor Embedding*) é uma técnica de redução de dimensionalidade que projeta esses vetores em um plano 2D, tentando preservar a vizinhança: pontos próximos em 512 dimensões tendem a continuar próximos no gráfico.

É uma ferramenta de visualização e diagnóstico, não faz parte do pipeline de comparação em si (a busca da Seção 3 já foi feita com os embeddings originais, em 512 dimensões). Mas serve para confirmar visualmente algo que os números de similaridade já sugeriram: os embeddings de uma mesma pessoa formam agrupamentos (*clusters*) coesos e separados dos demais.

### 4.1. Reduzindo a dimensionalidade

Juntamos os embeddings do banco com o embedding da consulta em uma única matriz antes de rodar o t-SNE, para que a consulta apareça projetada no mesmo espaço 2D. O parâmetro `perplexity` deve ser menor que o número de amostras; para o nosso banco pequeno (40 fotos + 1 consulta), usamos um valor baixo.

In [ ]:
embeddings_totais = np.vstack([embeddings_banco, embedding_consulta])

tsne = TSNE(n_components=2, perplexity=15, init='pca', random_state=42)
embeddings_2d = tsne.fit_transform(embeddings_totais)

embeddings_2d_banco = embeddings_2d[:-1]
embeddings_2d_consulta = embeddings_2d[-1]

### 4.2. Apresentando o espaço vetorial

In [ ]:
cores = plt.cm.tab10.colors

plt.figure(figsize=(8, 7))

for indice_pessoa, pessoa in enumerate(pessoas):
    mascara = np.array(nomes_banco) == pessoa
    plt.scatter(
        embeddings_2d_banco[mascara, 0],
        embeddings_2d_banco[mascara, 1],
        color=cores[indice_pessoa],
        label=pessoa.replace('_', ' '),
        s=60,
        edgecolors='white',
    )

plt.scatter(
    *embeddings_2d_consulta,
    color='black',
    marker='*',
    s=350,
    label='Consulta',
    edgecolors='white',
)

plt.legend()
plt.title("Embeddings faciais projetados em 2D (t-SNE)")
plt.xticks([])
plt.yticks([])
plt.show()

A estrela preta, a consulta, cai bem dentro de um dos agrupamentos, o mesmo resultado encontrado na busca por similaridade da Seção 3, agora visível geometricamente: identidades diferentes ocupam regiões distintas do espaço de embeddings.

## 5. Verificação 1:1 — Simulando o Onboarding

A busca 1:N da Seção 3 resolve a pergunta "quem é essa pessoa, dentre todas as cadastradas?". Mas no onboarding de um colaborador, o fluxo mais comum é outro: a empresa já sabe *quem* a pessoa alega ser (ela acabou de informar seu CPF ou matrícula, por exemplo) e só precisa confirmar que a selfie tirada na hora bate com a foto de referência já cadastrada para aquela pessoa. É uma **verificação 1:1**: comparar exatamente dois rostos e decidir entre aceitar ou recusar, não uma busca no banco inteiro.

### 5.1. Selfie x foto de referência

Comparamos a selfie de consulta com uma única foto do banco, a primeira cadastrada para a identidade que a pessoa alega ter.

In [ ]:
referencia_path = paths_banco[nomes_banco.index('Tony_Blair')]

img_referencia = cv2.imread(str(referencia_path))
faces_referencia = app.get(img_referencia)
rosto_referencia = max(faces_referencia, key=area_rosto)
embedding_referencia = rosto_referencia.normed_embedding

fig, eixos = plt.subplots(1, 2, figsize=(6, 4))
eixos[0].imshow(cv2.cvtColor(img_consulta, cv2.COLOR_BGR2RGB))
eixos[0].set_title("Selfie (consulta)")
eixos[0].axis('off')

eixos[1].imshow(cv2.cvtColor(img_referencia, cv2.COLOR_BGR2RGB))
eixos[1].set_title("Foto de referência\n(cadastro)")
eixos[1].axis('off')

plt.tight_layout()
plt.show()

### 5.2. Calculando a similaridade e aplicando um limiar

Assim como na Seção 3, a similaridade é o produto escalar entre os dois embeddings normalizados. A diferença é a decisão: em vez de rankear candidatos, aplicamos um **limiar (threshold)** fixo, calibrado previamente, para decidir automaticamente entre aceitar ou recusar a verificação.

Esse limiar é uma escolha de negócio, não um valor universal: um limiar mais alto reduz falsos positivos (aceitar por engano uma pessoa errada), ao custo de mais falsos negativos (recusar por engano a pessoa certa), e vice-versa.

In [ ]:
LIMIAR_VERIFICACAO = 0.45

similaridade_11 = float(np.dot(embedding_consulta, embedding_referencia))
verificado = similaridade_11 >= LIMIAR_VERIFICACAO

print(f"Similaridade: {similaridade_11:.3f} (limiar: {LIMIAR_VERIFICACAO})")
print("Identidade verificada!" if verificado else "Identidade não verificada.")

### 5.3. Apresentando o resultado

Por fim, uma apresentação típica de um app de onboarding: as duas fotos lado a lado, com o resultado da verificação.

In [ ]:
cor_resultado = (0, 200, 0) if verificado else (0, 0, 220)
texto_resultado = "VERIFICADO" if verificado else "NAO VERIFICADO"

def com_moldura(img: np.ndarray, cor: tuple, espessura: int = 12) -> np.ndarray:
    """ Adiciona uma moldura colorida ao redor da imagem """
    return cv2.copyMakeBorder(img, espessura, espessura, espessura, espessura, cv2.BORDER_CONSTANT, value=cor)

fig, eixos = plt.subplots(1, 2, figsize=(6, 4.5))
eixos[0].imshow(cv2.cvtColor(com_moldura(img_consulta, cor_resultado), cv2.COLOR_BGR2RGB))
eixos[0].set_title("Selfie (consulta)")
eixos[0].axis('off')

eixos[1].imshow(cv2.cvtColor(com_moldura(img_referencia, cor_resultado), cv2.COLOR_BGR2RGB))
eixos[1].set_title("Foto de referência\n(cadastro)")
eixos[1].axis('off')

fig.suptitle(f"{texto_resultado}  (similaridade={similaridade_11:.2f})", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Exemplo Negativo — Identidade Não Confere

A Seção 5 mostrou o caminho feliz: a pessoa é quem diz ser, e o sistema confirma. Mas um verificador só é útil se também souber dizer "não". Vamos repetir exatamente o mesmo fluxo de verificação 1:1, com uma diferença: em vez de comparar a selfie com a foto de referência da identidade correta (Tony Blair), comparamos com a foto de referência de outra pessoa do banco (George W. Bush), simulando alguém tentando concluir o onboarding, por engano ou má-fé, sob a identidade errada.

### 6.1. Selfie x foto de referência de outra identidade

In [ ]:
referencia_errada_path = paths_banco[nomes_banco.index('George_W_Bush')]

img_referencia_errada = cv2.imread(str(referencia_errada_path))
faces_referencia_errada = app.get(img_referencia_errada)
rosto_referencia_errada = max(faces_referencia_errada, key=area_rosto)
embedding_referencia_errada = rosto_referencia_errada.normed_embedding

fig, eixos = plt.subplots(1, 2, figsize=(6, 4))
eixos[0].imshow(cv2.cvtColor(img_consulta, cv2.COLOR_BGR2RGB))
eixos[0].set_title("Selfie (consulta)")
eixos[0].axis('off')

eixos[1].imshow(cv2.cvtColor(img_referencia_errada, cv2.COLOR_BGR2RGB))
eixos[1].set_title("Foto de referência\n(identidade alegada: George W. Bush)")
eixos[1].axis('off')

plt.tight_layout()
plt.show()

### 6.2. Calculando a similaridade

Reaproveitamos o mesmo `LIMIAR_VERIFICACAO` da Seção 5.2. A decisão automática deve seguir sempre o mesmo critério, não importa qual identidade está sendo alegada.

In [ ]:
similaridade_negativa = float(np.dot(embedding_consulta, embedding_referencia_errada))
verificado_negativo = similaridade_negativa >= LIMIAR_VERIFICACAO

print(f"Similaridade: {similaridade_negativa:.3f} (limiar: {LIMIAR_VERIFICACAO})")
print("Identidade verificada!" if verificado_negativo else "Identidade não verificada.")

### 6.3. Apresentando o resultado

In [ ]:
cor_resultado_negativo = (0, 200, 0) if verificado_negativo else (0, 0, 220)
texto_resultado_negativo = "VERIFICADO" if verificado_negativo else "NAO VERIFICADO"

fig, eixos = plt.subplots(1, 2, figsize=(6, 4.5))
eixos[0].imshow(cv2.cvtColor(com_moldura(img_consulta, cor_resultado_negativo), cv2.COLOR_BGR2RGB))
eixos[0].set_title("Selfie (consulta)")
eixos[0].axis('off')

eixos[1].imshow(cv2.cvtColor(com_moldura(img_referencia_errada, cor_resultado_negativo), cv2.COLOR_BGR2RGB))
eixos[1].set_title("Foto de referência\n(identidade alegada: George W. Bush)")
eixos[1].axis('off')

fig.suptitle(f"{texto_resultado_negativo}  (similaridade={similaridade_negativa:.2f})", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

Esse par de exemplos, um verificado e um recusado, com exatamente o mesmo código e o mesmo limiar, é o que torna a verificação 1:1 confiável em produção: a decisão não depende de julgamento caso a caso, só da similaridade calculada e do limiar definido antecipadamente. Fica ainda uma última pergunta em aberto: o quão confiável é esse limiar? Essa é a pergunta que testes com falsos positivos e falsos negativos, sobre um conjunto de validação maior, respondem antes de qualquer sistema desses ir para produção com dados reais.